# 02 — Quantum Circuit Visualization & Verification

Phần này phục vụ milestone Tuần 3:
1. Vẽ 3 QFL variants bằng `qml.draw()` / `qml.draw_mpl()`
2. Circuit specs (depth, gate counts) qua `qml.specs`
3. So sánh encoding strategies (Angle / Amplitude / IQP) — E8
4. Verify **parameter-shift == backprop** (gradient correctness)
5. Demo NISQ noise simulation (depolarizing)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pennylane as qml
from src.quantum.quantum_fusion import (
    _tensor_circuit,
    _attention_kernel,
    _interference_circuit,
    QuantumFusionTensor,
    QuantumFusionInterference,
)
from src.quantum.encoding import ENCODINGS, get_encoding, estimate_encoding_cost

print("PennyLane version:", qml.__version__)

## 1. QFL-Tensor — text-based circuit diagram

In [ ]:
text_feat = torch.tensor([0.5, -0.2, 0.8, 0.1])
image_feat = torch.tensor([0.3, 0.6, -0.4, 0.9])
weights = torch.randn(3, 16) * 0.01

print(qml.draw(_tensor_circuit)(text_feat, image_feat, weights))

In [ ]:
# Matplotlib diagram -> paper/figures/circuit_diagram.png
fig_dir = PROJECT_ROOT / "paper" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

try:
    fig, ax = qml.draw_mpl(_tensor_circuit)(text_feat, image_feat, weights)
    fig.suptitle("QFL-Tensor: 8-qubit fusion circuit")
    fig.savefig(fig_dir / "circuit_diagram.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_dir / "circuit_diagram.png")
except Exception as e:
    print("draw_mpl failed:", e)

## 2. Circuit specs — depth & gate counts

In [ ]:
specs = qml.specs(_tensor_circuit)(text_feat, image_feat, weights)
print("Depth       :", specs["resources"].depth)
print("Total gates :", specs["resources"].gate_types)
print("Trainable params:", specs["num_trainable_tape_parameters"])

In [ ]:
# Specs cho attention kernel và interference circuit
for name, fn, args in [
    ("Attention kernel (4q)", _attention_kernel,
     (torch.randn(4), torch.randn(4), torch.randn(4) * 0.01)),
    ("Interference (8q)", _interference_circuit,
     (torch.randn(4), torch.randn(4), torch.randn(8) * 0.01)),
]:
    s = qml.specs(fn)(*args)
    print(f"{name}: depth={s['resources'].depth}, gates={dict(s['resources'].gate_types)}")

## 3. Encoding comparison (E8 prep)

Ba chiến lược encode features classical → quantum:

In [ ]:
features = np.array([0.4, -0.7, 0.2, 0.9])
wires = list(range(4))

for name in ["angle", "iqp"]:
    enc_fn = get_encoding(name)

    def demo_circuit(f):
        enc_fn(f, wires)
        return qml.state()

    dev = qml.device("default.qubit", wires=4)
    node = qml.QNode(demo_circuit, dev)
    state = node(torch.tensor(features))
    probs = np.abs(state.detach().numpy()) ** 2 if hasattr(state, "detach") else np.abs(state) ** 2

    plt.figure(figsize=(9, 3))
    plt.bar(range(len(probs)), probs)
    plt.title(f"{name} encoding: state probabilities")
    plt.xlabel("basis state index")
    plt.tight_layout()
    plt.show()

# Chi phí gate cho từng encoding (E10/E8 analysis)
for enc in ["angle", "amplitude", "iqp"]:
    print(f"{enc:>10}: {estimate_encoding_cost(enc, n_features=4)}")

## 4. Gradient verification: parameter-shift vs backprop

Milestone Tuần 3 yêu cầu hai diff methods cho cùng kết quả.

In [ ]:
dev_ps = qml.device("default.qubit", wires=8)

def make_qnode(diff_method):
    @qml.qnode(dev_ps, interface="torch", diff_method=diff_method)
    def circuit(text_emb, image_emb, weights):
        return _tensor_circuit(text_emb, image_emb, weights)
    return circuit


text_in = torch.randn(4, requires_grad=False)
image_in = torch.randn(4, requires_grad=False)
w = torch.randn(3, 16, requires_grad=True)

losses = {}
grads = {}
for method in ["backprop", "parameter-shift"]:
    node = make_qnode(method)
    out = torch.stack(node(text_in, image_in, w))
    loss = out.sum()
    loss.backward()
    losses[method] = loss.item()
    grads[method] = w.grad.clone()
    w.grad = None

diff_loss = abs(losses["backprop"] - losses["parameter-shift"])
diff_grad = (grads["backprop"] - grads["parameter-shift"]).abs().max().item()
print(f"Loss difference      : {diff_loss:.2e}")
print(f"Max gradient element |Δ|: {diff_grad:.2e}")
assert diff_grad < 1e-5, "Gradients do not match!"
print("PASS ✓ parameter-shift ≡ backprop")

## 5. Gradient flow qua full module (autograd end-to-end)

In [ ]:
model = QuantumFusionTensor(n_qubits=8, n_layers=3)
t_batch = torch.randn(4, 4, requires_grad=True)
i_batch = torch.randn(4, 4, requires_grad=True)
out = model(t_batch, i_batch)
loss = out.sum()
loss.backward()

print("Output shape        :", tuple(out.shape))
print("weights grad norm   :", model.weights.grad.norm().item())
print("input grad exists   :", t_batch.grad is not None)
assert out.shape == (4, 8)
print("PASS ✓ gradient flows through PQC")

## 6. NISQ noise simulation (E9 prep)

Thêm depolarizing channel vào circuit trên `default.mixed` device.

In [ ]:
def noisy_expectation(prob):
    dev = qml.device("default.mixed", wires=2)

    @qml.qnode(dev, interface="torch")
    def circuit(x):
        qml.RY(x[0], wires=0)
        qml.RY(x[1], wires=1)
        qml.CNOT(wires=[0, 1])
        # Simulate depolarizing noise on both wires after gates
        for wire in range(2):
            qml.DepolarizingChannel(prob, wires=wire)
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

    return circuit

x = torch.tensor([0.6, 0.4])
noise_levels = [0.0, 0.005, 0.01, 0.02, 0.05]
vals = []
for p in noise_levels:
    v = noisy_expectation(p)(x).item()
    vals.append(v)
    print(f"p={p:<6} <ZZ> = {v:+.4f}")

plt.figure(figsize=(8, 4))
plt.plot(noise_levels, vals, "o-", color="#2196F3")
plt.xlabel("Depolarizing probability p")
plt.ylabel(r"$\langle Z_0 Z_1 \rangle$")
plt.title("Expectation decay under depolarizing noise (NISQ sim)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "noise_demo.png", dpi=150, bbox_inches="tight")
plt.show()